# Community detection (2026)

Utrecht Summer School on Network Science — Session 2


**The question for this morning:**

> *My network looks like it has groups — is that real, and what are they?*

On Tuesday we built a ladder of null models and ended on rung 3: *is there block structure
beyond degree heterogeneity?* Today is that rung, done properly.

One thing "looks like it has groups" is **not** evidence for: a force-directed layout.
Layout algorithms *manufacture* visual clusters — they pull connected nodes together by
design, so even a completely random graph draws as blobs. If your evidence for communities
is that the picture has lumps, you have no evidence yet.

**Today's workflow** (this is the roadmap for the morning, and for your own data this
afternoon):

```
1. Modularity maximization        — fast exploratory first look
        │
2. Null test                      — is the modularity value actually high?
   (Tuesday's configuration-model recipe, with modularity as the statistic)
   ├─ no  → maybe no real structure. The SBM can confirm: it is able
   │        to answer "one block".
   └─ yes → continue
        │
3. SBM inference                  — the principled version of the question
   If modularity and the SBM disagree on the groups → trust the SBM.
        │
4. Plain vs degree-corrected SBM  — model selection via description length
   (entropy: lower = better model, no ground truth needed)
        │
5. Validate                       — BESTest against an attribute (Tuesday!),
   or nested SBM if you suspect structure at several scales.
```

The line to remember: **modularity maximization is a fast heuristic, not a hypothesis
test** — it always returns an answer, whether or not there was a real question. The SBM
machinery exists because "does my network have communities?" deserves a real test.

Open in Colab:
[![Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jgarciab/NetworkScience/blob/main/Practicals/day3a_community_detection/Community_detection_2026.ipynb)

**You cannot save changes in this notebook — save a copy first: `File` → `Save a copy in
Drive`.**


## 0 — Setup

Same environment as Tuesday. The first cell **restarts the kernel** — expected. If a crash
message appears, just run the notebook again.


In [ ]:
!pip install -q condacolab
import condacolab
# Workaround issue with Python 3.12:
# condacolab.install()
condacolab.install_from_url("https://github.com/conda-forge/miniforge/releases/download/25.3.1-0/Miniforge3-Linux-x86_64.sh")

In [ ]:
!mamba install -q graph-tool

In [ ]:
# Workaround issue with Python 3.12
!mamba install -q scipy

In [ ]:
import graph_tool.all as gt
from graph_tool import topology, inference, generation, stats, correlations
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np

*Working on your own data this afternoon? Your copy of the BYOD starter notebook from
Tuesday still has your network loaded — keep working there, Part 2 is today's half.*


## 1 — Warm-up: what modularity actually measures (15 min)

Modularity compares the fraction of edges *inside* groups against what a degree-preserving
random graph would put inside those same groups:

$$Q = \sum_r \left( \frac{e_{rr}}{m} - \left(\frac{k_r}{2m}\right)^2 \right)$$

where, for each group $r$: $e_{rr}$ is the number of edges with both ends in $r$, $k_r$ is
the total degree of the group's nodes, and $m$ is the total number of edges. The first term
is what you observe; the second is what the configuration model expects (Tuesday, rung 2 —
modularity has the null *built in*).

**Exercise 1 — one calculation by hand, to fix the definition.** For the 9-node network
below, with the two groups given by the colours (orange: nodes 0–4, teal: nodes 5–8),
compute $Q$. You need three numbers per group. Show your working.

![https://github.com/piratepeel/piratepeel.github.io/raw/master/teaching/network9nodes.png](https://github.com/piratepeel/piratepeel.github.io/raw/master/old/teaching/network9nodes.png)


Now verify your number: build the network and let graph-tool compute the modularity
of the given partition.


In [ ]:
g9 = gt.Graph(directed=False)
g9.add_vertex(9)
# add the edges you see in the figure (nodes are numbered 0-8)
g9.add_edge_list([# insert code here
                  ])
# build the partition as an integer vertex property (0 = orange, 1 = teal)
b = g9.new_vertex_property('int')
# insert code here

print('Modularity of the 2-group partition:', gt.modularity(g9, b))

$Q = 0.3$ — is that... good? High? Significant? **You cannot tell from the number
alone.** The rest of the morning is about that problem: first a failure mode where
modularity maximization *undercounts* obvious structure, then one where it *invents*
structure that isn't there.


## 2 — Failure mode 1: the resolution limit

### Exercise 2

**(a)** Write a function `create_graph_k_cliques(k)` that generates a network of $k$
weakly-connected cliques of size 10 (each clique connected to the next by a single edge,
in a ring). Use it to generate a graph with $k = 10$ and draw it. If any structure ever
deserved to be called "10 obvious communities", it is this.

**(b)** Generate networks with $k \in \{10, 20, \ldots, 150\}$ cliques and use
modularity maximization to detect communities in each:

```python
state = gt.minimize_blockmodel_dl(g, state=inference.ModularityState)
```

Plot the number of communities modularity finds against the true number $k$. What do you
observe? Can you explain why this happens?


In [ ]:
def create_graph_k_cliques(k, clique_size=10):
    g = gt.Graph(directed=False)
    # insert code here
    return g

g = create_graph_k_cliques(k=10)
gt.graph_draw(g)

In [ ]:
k_values = range(10, 160, 10)
n_groups = []

for k in k_values:
    g = create_graph_k_cliques(k=k)
    # run modularity maximization and record the number of groups found
    # insert code here

# plot groups found vs the true number of cliques k
# insert code here

## 3 — Modularity vs statistical inference: three networks

Now the core of the morning: the same two methods — modularity maximization and SBM
inference — on three networks where **we know the truth**, because we generate them.

### 3a — Network One: planted communities

Ten groups of ~100 nodes, dense inside, sparse between. Ground truth exists (`bm` below)
because we planted it.


In [ ]:
def prob(a, b):
    if a == b:
        return 0.9
    else:
        return 0.01

g, bm = generation.random_graph(1000, lambda: np.random.poisson(10),
                                directed=False,
                                model="blockmodel",
                                block_membership=lambda: np.random.randint(10),
                                edge_probs=prob)

Draw it with a force-directed layout, coloured by the *true* community assignment:


In [ ]:
pos = gt.arf_layout(g)
gt.graph_draw(g, pos=pos, vertex_fill_color=bm)

Of course, when we encounter networks in the wild we don't know the assignments a
priori. Let's see if we can recover them. First, modularity maximization:


In [ ]:
state = gt.minimize_blockmodel_dl(g, state=inference.ModularityState)
modularity_g = state.modularity()
print('Maximised modularity:', modularity_g)
print('Number of groups:', state.get_B())
state.draw(pos=pos)

Maximising modularity seems to work well!

We can also use statistical inference with the SBM — instead of maximizing a quality score,
this asks: *what block structure makes this network's edges most probable?*


In [ ]:
state = gt.minimize_blockmodel_dl(g, multilevel_mcmc_args={'niter': 5})
print('Number of groups:', state.get_B())
state.draw(pos=pos)

Both methods recover the planted structure. So which should we use?

### 3b — Network Two: no communities at all

The same recipe, minus the planting: a pure Erdős–Rényi random graph. There is, by
construction, **nothing to find**.


In [ ]:
g = generation.random_graph(1000, lambda: np.random.poisson(10), directed=False)
state = gt.minimize_blockmodel_dl(g, state=inference.ModularityState)
modularity_g = state.modularity()
print('Maximised modularity:', modularity_g)
print('Number of groups:', state.get_B())
pos = gt.arf_layout(g)
state.draw(pos=pos)

**What do you notice here? Is this what you would expect?**

**Are these communities meaningful?**

Modularity confidently reports a partition with a respectable-looking $Q$ — on a network
with no structure whatsoever. (And notice how the layout obligingly draws the coloured
groups as clumps.) Never trust a bare modularity number; test it. This is **Tuesday's
Exercise 2 recipe verbatim** — configuration-model rewiring — with modularity as the
statistic:


In [ ]:
g_rand = g.copy()

n_samples = 1000
modularity_values = np.empty(n_samples)
for i in range(n_samples):
    generation.random_rewire(g_rand, model='configuration')
    state = gt.minimize_blockmodel_dl(g_rand, state=inference.ModularityState)
    modularity_values[i] = state.modularity()

In [ ]:
pvalue = np.mean(modularity_values >= modularity_g)
plt.hist(modularity_values, bins=50, label='configuration null')
plt.axvline(modularity_g, color='black', label='observed')
plt.legend()
plt.xlabel('Maximised modularity')
plt.show()
print(f'p-value: {pvalue:.3f}')

**What do these results tell you?**

**What happens when you use the SBM to infer the communities?**


In [ ]:
state = gt.minimize_blockmodel_dl(g, multilevel_mcmc_args={'niter': 5})
print('Number of groups:', state.get_B())
state.draw(pos=pos)

### 3c — Network Three: a needle of structure in a haystack of noise

A slight modification of Network Two...


In [ ]:
g = generation.random_graph(1000, lambda: np.random.poisson(10), directed=False)

clique_size = 25
for i in range(clique_size):
    for j in range(i + 1, clique_size):
        g.add_edge(i, j, False)

state = gt.minimize_blockmodel_dl(g, state=inference.ModularityState)
modularity_g = state.modularity()
print('Maximised modularity:', modularity_g)
print('Number of groups:', state.get_B())
pos = gt.arf_layout(g)
state.draw(pos=pos)

**What is the modification?**

We can try the null hypothesis test again...


In [ ]:
g_rand = g.copy()

n_samples = 1000
modularity_values = np.empty(n_samples)
for i in range(n_samples):
    generation.random_rewire(g_rand, model='configuration')
    state = gt.minimize_blockmodel_dl(g_rand, state=inference.ModularityState)
    modularity_values[i] = state.modularity()

In [ ]:
pvalue = np.mean(modularity_values >= modularity_g)
plt.hist(modularity_values, bins=50, label='configuration null')
plt.axvline(modularity_g, color='black', label='observed')
plt.legend()
plt.xlabel('Maximised modularity')
plt.show()
print(f'p-value: {pvalue:.3f}')

**How can you interpret this result?**

**Now try using the SBM... What do you notice?**


In [ ]:
state = gt.minimize_blockmodel_dl(g, multilevel_mcmc_args={'niter': 5})
print('Number of groups:', state.get_B())
state.draw(pos=pos)

## 4 — Degree correction, and how to choose a model without ground truth

We can think of the plain SBM as a collection of ER random graphs: every node in a block
has the same expected degree. Tuesday's degree histograms tell you how realistic that is —
real networks have hubs and peripheral nodes *inside* the same community. The
**degree-corrected SBM (DC-SBM)** fixes this by treating each node's degree as given
(rung 2 thinking, applied inside the model).

Our test case: the classic `polblogs` network — hyperlinks between US political blogs in
2004, which famously splits into a liberal and a conservative camp. That known division is
stored in the `value` attribute, but **pretend for now that we don't have it** — that's the
situation you'll be in with your own data.

To make the comparison sharp we force both models to find exactly 2 groups (`B_max=2`) and
ask: *which two groups does each model choose?*


In [ ]:
g_polblogs = gt.collection.data['polblogs']
g_polblogs = topology.extract_largest_component(g_polblogs, prune=True)
print(g_polblogs.num_vertices(), 'blogs,', g_polblogs.num_edges(), 'links')
gt.graph_draw(g_polblogs, pos=g_polblogs.vp['pos'])

In [ ]:
state_plain = gt.minimize_blockmodel_dl(g_polblogs, state_args={'deg_corr': False},
                                        multilevel_mcmc_args={'niter': 5, 'B_max': 2})
print('Entropy:', state_plain.entropy())
print('Number of groups:', state_plain.get_B())
state_plain.draw(pos=g_polblogs.vp['pos'])

In [ ]:
state_dc = gt.minimize_blockmodel_dl(g_polblogs, state_args={'deg_corr': True},
                                     multilevel_mcmc_args={'niter': 5, 'B_max': 2})
print('Entropy:', state_dc.entropy())
print('Number of groups:', state_dc.get_B())
state_dc.draw(pos=g_polblogs.vp['pos'])

**Exercise 3 — you have two models and no ground truth. Which do you report, and
why?**

The two partitions clearly differ (look at the two drawings — the layout places the two
political camps left and right). Your evidence, without peeking at any attribute, is the
**entropy** printed above. Entropy here is a *description length*: the number of bits
needed to describe the network using the model. A model with lower entropy compresses the
data better — it is a better model, by a criterion (MDL) that automatically penalises
complexity, so it needs no ground truth and no held-out data.

**(a)** Compare the two entropies. Which model wins, and by how many bits?

**(b)** Look at the two drawings: describe in words what kind of "groups" the losing model
found.


In [ ]:
# (a) compare the entropies of state_plain and state_dc
# insert code here

# (b) describe in words what the losing model's two groups are
#     (write your answer in the markdown cell below or discuss with your neighbour)

**Exercise 4 — now peek.** The `value` vertex property holds the known political
leaning (0/1). For each of the two fitted models, check how well its 2-group partition
agrees with the political division — e.g. via `gt.partition_overlap`, or simply by drawing
the network coloured by `value` and comparing with the drawings above.

Does the MDL verdict from Exercise 3 — reached *without* the labels — agree with what the
labels say?


In [ ]:
b_true = g_polblogs.vp['value']
# draw the network coloured by the true political leaning
# insert code here

# compare each fitted partition against b_true (hint: gt.partition_overlap)
# insert code here

**Exercise 5 — release the constraint.** `B_max=2` was for illustration. Remove it
and let each model choose its own number of groups. What does each choose, and does the
entropy comparison still favour degree correction?


In [ ]:
# fit both models again without B_max, compare group counts and entropies
# insert code here

## 5 — Optional extension: hierarchical communities

*Do this section if you suspect structure at more than one scale (e.g.
departments-within-divisions) — otherwise skip ahead to the practical; you can always come
back.*

Here is the full face-to-face contact network of the high school from Tuesday. Students
attend classes, classes group into subject streams — structure at several scales.


In [ ]:
!wget https://networks.skewed.de/net/sp_high_school/files/proximity.gt.zst
g = gt.load_graph("proximity.gt.zst")
gt.remove_parallel_edges(g)

In [ ]:
# class labels are stored in the following vertex property
set(g.vp['class'])

In [ ]:
# colour the nodes by the class they attend
subjects = list(set(g.vp['class']))
subjects.sort()
scmap = plt.get_cmap('tab20c')

subjectcolormap = dict(zip(subjects, scmap(list(range(len(subjects))))))

vertex_color = g.new_vertex_property("vector<float>")
for v in g.vertices():
    vertex_color[v] = subjectcolormap[g.vp['class'][v]]
g.vp.vertex_color = vertex_color

# plot the network
gt.graph_draw(g, vertex_fill_color=g.vp['vertex_color'])

# create legend
fig = plt.figure(figsize=(1, 1))
for subject in subjects:
    plt.plot([1], [1], label=subject, marker='s', color=subjectcolormap[subject])
lg = plt.legend(fontsize='xx-large', markerscale=3, ncol=3)
out = plt.axis('off')

In [ ]:
# fit hierarchical (nested) blockmodel
state = gt.minimize_nested_blockmodel_dl(g)

# plot the hierarchy
state.draw(vertex_fill_color=g.vp['vertex_color'])
state.print_summary()

# create legend
fig = plt.figure(figsize=(1, 1))
for subject in subjects:
    plt.plot([1], [1], label=subject, marker='s', color=subjectcolormap[subject])
lg = plt.legend(fontsize='xx-large', markerscale=3, ncol=3)
out = plt.axis('off')

print('Entropy:', state.entropy())

**Discuss:** read the `print_summary()` output bottom-up. Do the lowest-level blocks
correspond to school classes (colours)? What do the upper levels group together?


## 6 — Practical time: the full pipeline on a real network

Run the whole workflow from the framing section end to end, with instructors circulating:

1. **Modularity maximization** — exploratory first look.
2. **Null test** — is the modularity value actually high? (configuration rewiring)
3. **SBM inference** — group count and partition; if it disagrees with modularity, you
   know which to trust.
4. **Plain vs DC-SBM** — entropy comparison (check the degree histogram first).
5. **Validate** — BESTest against a node attribute (Tuesday's Section 6 — the
   `random_entropy` function and, for string attributes, `categorical_to_int`).

**Work in pairs, pick a dataset:**

- **Option 1 — American college football** (`gt.collection.data['football']`): games
  between US college teams in the 2000 season; the `value` attribute holds each team's
  conference. Known-but-imperfect truth (independent teams, one oddly-assigned
  conference), and 12 small communities — should the resolution limit worry you here?
- **Option 2 — Political books** (`gt.collection.data['polbooks']`): co-purchasing of US
  politics books; `value` holds liberal / conservative / neutral. The "neutral" category
  makes validation genuinely ambiguous — is it a community or a label of convenience?
- **Option 3 — Tuesday's dataset:** rerun the pipeline on whichever Block-3 dataset you
  used yesterday (faculty hiring, the Dutch classroom, or the trade layer). Your rung-3
  DC-SBM fit was a preview of exactly this.

**And as soon as your own data is ready, switch to it** — Part 2 of your BYOD notebook is
this pipeline, and your network is still loaded from Tuesday.


In [ ]:
# the BESTest null, from Tuesday
def random_entropy(g, n_groups):
    n = g.num_vertices()
    b = np.random.randint(n_groups, size=n)
    vprop_b = g.new_vertex_property("int")
    for i in range(n):
        vprop_b[g.vertex(i)] = b[i]
    state = gt.BlockState(g, b=vprop_b)
    return state.entropy()

def categorical_to_int(g, prop_name):
    """Map a categorical vertex property to an int vertex property (0..K-1)."""
    labels = [g.vp[prop_name][v] for v in g.vertices()]
    cats = sorted(set(labels))
    mapping = {c: i for i, c in enumerate(cats)}
    b = g.new_vertex_property('int')
    for v in g.vertices():
        b[v] = mapping[g.vp[prop_name][v]]
    return b, mapping

### Option 1 — college football

In [ ]:
g_fb = gt.collection.data['football']
print(g_fb)
print('Node attributes:', list(g_fb.vp.keys()))
print('Conferences:', len(set(g_fb.vp['value'])))

In [ ]:
# Step 1 -- modularity maximization: Q and number of groups
# insert code here

In [ ]:
# Step 2 -- null test: does the modularity beat the configuration null?
# insert code here

In [ ]:
# Step 3 -- SBM inference: does the group count agree with Step 1?
# insert code here

In [ ]:
# Step 4 -- plain vs DC-SBM entropy (look at the degree histogram first)
# insert code here

In [ ]:
# Step 5 -- validate against the conference attribute (BESTest)
# insert code here

### Option 2 — political books

In [ ]:
g_pb = gt.collection.data['polbooks']
print(g_pb)
print('Node attributes:', list(g_pb.vp.keys()))
print('Labels:', set(g_pb.vp['value']))

In [ ]:
# Steps 1-5, same pipeline. When you reach Step 5: 'value' holds strings,
# so convert with categorical_to_int first. Is "neutral" a community?
# insert code here

## 7 — The two mornings, in one box

**Tuesday:** *is what I'm seeing surprising?* — the null-model ladder
(ER → configuration → DC-SBM), plus the BESTest for attributes.

**Today:** *are the groups real, and what are they?* — modularity as a fast look, the null
test to keep it honest, the SBM to do the real inference, description length to choose
between models, and the BESTest again to connect structure back to what you know about
your nodes.

Same recipe throughout: **never interpret a number without asking what a suitably boring
random network would have given you.** That habit is the thing to take home — the specific
tools are just its implementation.
